In [ ]:
import mercury as mr
app = mr.App(
    title='¿Quién mueve los hilos de la ciencia mexicana?',
    description='Redes de coautoría científica en México 2018–2024',
    show_code=False
)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df_nodos      = pd.read_csv('data/clean/nodos_autores.csv')
df_aristas    = pd.read_csv('data/clean/aristas_colaboraciones.csv')
df_autorias   = pd.read_csv('data/clean/autorias_limpias.csv')
df_pos        = pd.read_csv('data/clean/posiciones.csv')
df_central    = pd.read_csv('data/clean/centralidad.csv')
df_nodos_sub  = pd.read_csv('data/clean/nodos_sub.csv')
df_aristas_sub= pd.read_csv('data/clean/aristas_sub.csv')
df_inst       = pd.read_csv('data/clean/resumen_instituciones.csv')
df_inter      = pd.read_csv('data/clean/colabs_inter_inst.csv')
df_evol_tipo  = pd.read_csv('data/clean/evolucion_tipo.csv')
df_evol_colab = pd.read_csv('data/clean/evolucion_colabs.csv')

# Paleta oscura con acentos morado/azul
BG = '#0d1117'
CARD = '#161b22'
TEXT = '#e6edf3'
GRID = '#21262d'
ACCENT1 = '#8b5cf6'  # morado
ACCENT2 = '#3b82f6'  # azul
ACCENT3 = '#06b6d4'  # cyan
ACCENT4 = '#10b981'  # verde
ACCENT5 = '#f59e0b'  # amber
ACCENT6 = '#ef4444'  # rojo
ACCENT7 = '#ec4899'  # rosa
PALETTE = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, ACCENT5, ACCENT6, ACCENT7, '#6366f1', '#14b8a6', '#f97316']

dark_template = go.layout.Template()
dark_template.layout = go.Layout(
    paper_bgcolor=BG,
    plot_bgcolor=CARD,
    font=dict(color=TEXT, family='Inter, system-ui, sans-serif'),
    title=dict(font=dict(size=18, color=TEXT)),
    xaxis=dict(gridcolor=GRID, zerolinecolor=GRID),
    yaxis=dict(gridcolor=GRID, zerolinecolor=GRID),
    colorway=PALETTE,
    margin=dict(l=40, r=40, t=60, b=40),
)

# ¿Quién mueve los hilos de la ciencia mexicana?

Cada artículo científico es una conversación. Cuando esas conversaciones se acumulan durante años, forman **redes invisibles** que definen el poder y alcance de la ciencia de un país. Este análisis explora la red de coautoría científica en México entre **2018 y 2024**, usando datos abiertos de [OpenAlex](https://openalex.org/).

---

### 📊 Panorama general

In [ ]:
n_inv = len(df_nodos)
n_colabs = len(df_aristas)
n_inst = df_nodos['inst_principal'].nunique()
n_works = df_autorias['work_id'].nunique()

fig_stats = go.Figure()
vals = [n_inv, n_works, n_colabs, n_inst]
labels = ['Investigadores', 'Artículos', 'Colaboraciones', 'Instituciones']
colors = [ACCENT1, ACCENT2, ACCENT3, ACCENT4]
icons = ['👩\u200d🔬', '📄', '🤝', '🏛️']

for i, (v, l, c, ic) in enumerate(zip(vals, labels, colors, icons)):
    fig_stats.add_trace(go.Indicator(
        mode='number',
        value=v,
        title=dict(text=f'{ic} {l}', font=dict(size=14, color=TEXT)),
        number=dict(font=dict(size=36, color=c)),
        domain=dict(x=[i/4, (i+1)/4], y=[0, 1]),
    ))
fig_stats.update_layout(
    template=dark_template, height=120,
    margin=dict(l=10, r=10, t=10, b=10),
)
fig_stats.show()

---
## 🏛️ ¿Quién domina la producción científica?

El siguiente treemap muestra las **15 instituciones con más investigadores** en la red. El tamaño del bloque representa la cantidad de investigadores; el color indica las **citas acumuladas promedio** — instituciones más oscuras tienen mayor impacto citado por artículo.

In [ ]:
top15 = df_inst.head(15).copy()
top15['nombre_corto'] = top15['inst_principal'].apply(
    lambda x: x[:35] + '…' if len(str(x)) > 35 else x
)
top15['label'] = top15.apply(
    lambda r: f"{r['nombre_corto']}<br>{r['num_investigadores']} inv. · {r['total_citas']:,.0f} citas", axis=1
)

fig1 = px.treemap(
    top15,
    path=['label'],
    values='num_investigadores',
    color='promedio_citas',
    color_continuous_scale=[[0, '#1e1b4b'], [0.5, ACCENT1], [1, '#c4b5fd']],
    hover_data={'total_articulos': True, 'total_citas': True, 'promedio_citas': ':.0f'},
)
fig1.update_layout(
    template=dark_template, height=500,
    title='Top 15 instituciones por investigadores en la red',
    coloraxis_colorbar=dict(title='Citas prom.', tickfont=dict(color=TEXT)),
    margin=dict(l=10, r=10, t=60, b=10),
)
fig1.update_traces(
    textfont=dict(color='white', size=11),
    marker=dict(cornerradius=5),
)
fig1.show()

---
## 📈 ¿Cómo ha evolucionado la ciencia mexicana?

Las áreas apiladas muestran el **número de publicaciones por año**, separadas por tipo de institución. La mayoría de la producción viene de **universidades** (education), pero los centros gubernamentales y de salud también contribuyen significativamente.

In [ ]:
tipo_labels = {
    'education': 'Universidades',
    'facility': 'Centros de inv.',
    'government': 'Gobierno',
    'nonprofit': 'Sin fines de lucro',
    'healthcare': 'Salud',
    'company': 'Empresas',
    'funder': 'Fondos',
    'other': 'Otros',
    'archive': 'Archivos',
}
df_ev = df_evol_tipo.copy()
df_ev['tipo_es'] = df_ev['inst_tipo'].map(tipo_labels).fillna(df_ev['inst_tipo'])

fig2 = px.area(
    df_ev, x='anio', y='publicaciones', color='tipo_es',
    labels={'anio': 'Año', 'publicaciones': 'Publicaciones', 'tipo_es': 'Tipo'},
    title='Producción científica por tipo de institución',
    color_discrete_sequence=PALETTE,
)
fig2.update_layout(
    template=dark_template, height=420,
    xaxis=dict(tickmode='linear', dtick=1),
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center', font=dict(size=11)),
)
fig2.update_traces(line=dict(width=0.5))
fig2.show()

---
## 🕸️ La red de coautoría

Cada punto es un investigador. Cada línea es una colaboración publicada. Los colores representan **clusters temáticos** — grupos de investigadores que publican juntos frecuentemente. Los puntos más grandes indican mayor centralidad en la red (más conexiones). Pasa el cursor sobre un punto para ver su nombre e institución.

In [ ]:
df_plot = df_nodos_sub.merge(df_pos, on='autor_id')
df_plot = df_plot.merge(
    df_central[['autor_id', 'degree', 'betweenness']], on='autor_id', how='left'
)
df_plot['degree'] = df_plot['degree'].fillna(1)

# Asignar colores por institución top, resto gris
top_inst = df_plot['institucion'].value_counts().head(8).index.tolist()
inst_color_map = {inst: PALETTE[i] for i, inst in enumerate(top_inst)}
df_plot['color'] = df_plot['institucion'].map(inst_color_map).fillna('#444444')
df_plot['inst_short'] = df_plot['institucion'].apply(lambda x: x[:40] + '…' if len(str(x)) > 40 else x)

# Aristas
df_e = df_aristas_sub.merge(
    df_pos.rename(columns={'autor_id':'source','x':'x0','y':'y0'}), on='source')
df_e = df_e.merge(
    df_pos.rename(columns={'autor_id':'target','x':'x1','y':'y1'}), on='target')

edge_x, edge_y = [], []
for _, r in df_e.iterrows():
    edge_x += [r.x0, r.x1, None]
    edge_y += [r.y0, r.y1, None]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.15, color='rgba(139,92,246,0.15)'),
    hoverinfo='none', showlegend=False,
))

fig3.add_trace(go.Scatter(
    x=df_plot['x'], y=df_plot['y'], mode='markers',
    marker=dict(
        size=[max(3, min(14, d**0.7 * 2)) for d in df_plot['degree']],
        color=df_plot['color'],
        opacity=0.9,
        line=dict(width=0.3, color='rgba(255,255,255,0.3)'),
    ),
    text=['<b>' + n + '</b><br>' + i + '<br>' + str(a) + ' arts · ' + str(c) + ' citas'
          for n, i, a, c in zip(df_plot['nombre'], df_plot['inst_short'], df_plot['articulos'], df_plot['citas'])],
    hovertemplate='%{text}<extra></extra>',
    showlegend=False,
))

fig3.update_layout(
    template=dark_template, height=600,
    title=f'Red de coautoría — {len(df_plot)} investigadores más conectados',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False),
    plot_bgcolor=BG,
)
fig3.show()

---
## 🏆 Los puentes de la ciencia

La **centralidad de intermediación** mide cuántas veces un investigador aparece en el camino más corto entre dos colegas. No es necesariamente el más prolífico — es el **conector**, el que une comunidades que de otra forma estarían aisladas. El largo de la barra indica su centralidad; el color su institución.

In [ ]:
top20 = df_central[df_central['betweenness'] > 0].head(20).copy()
top20 = top20.sort_values('betweenness', ascending=True)
top20['inst_short'] = top20['institucion'].apply(lambda x: x[:35] + '…' if len(str(x)) > 35 else x)

fig4 = go.Figure()
# Lollipop lines
for _, r in top20.iterrows():
    fig4.add_trace(go.Scatter(
        x=[0, r['betweenness']], y=[r['nombre'], r['nombre']],
        mode='lines', line=dict(color=GRID, width=1.5),
        showlegend=False, hoverinfo='none',
    ))

# Dots
fig4.add_trace(go.Scatter(
    x=top20['betweenness'], y=top20['nombre'],
    mode='markers+text',
    marker=dict(
        size=top20['citas'].apply(lambda c: max(8, min(22, c**0.35))),
        color=ACCENT1,
        line=dict(width=1, color='rgba(255,255,255,0.4)'),
    ),
    text=[f" {r['inst_short']}" for _, r in top20.iterrows()],
    textposition='middle right',
    textfont=dict(size=9, color='#9ca3af'),
    hovertemplate='<b>%{y}</b><br>Centralidad: %{x:.4f}<br>Citas: %{customdata[0]:,}<br>Artículos: %{customdata[1]}<extra></extra>',
    customdata=top20[['citas', 'articulos']].values,
    showlegend=False,
))

fig4.update_layout(
    template=dark_template, height=500,
    title='Top conectores — Centralidad de intermediación',
    xaxis=dict(title='Centralidad de intermediación', showgrid=True),
    yaxis=dict(showgrid=False),
    plot_bgcolor=BG,
)
fig4.show()

---
## 🤝 ¿Cómo se conectan las instituciones?

El siguiente diagrama Sankey muestra los **flujos de colaboración entre las 12 instituciones más conectadas**. Las bandas más gruesas indican más colaboraciones conjuntas entre investigadores de ambas instituciones. Revela qué alianzas interinstitucionales sostienen la ciencia mexicana.

In [ ]:
# Top 12 instituciones por colaboraciones inter-institucionales
all_inst = pd.concat([
    df_inter['inst_source'].rename('inst'),
    df_inter['inst_target'].rename('inst'),
])
top12_inst = (
    df_inter.melt(id_vars=['total_colabs'], value_vars=['inst_source','inst_target'], value_name='inst')
    .groupby('inst')['total_colabs'].sum()
    .sort_values(ascending=False)
    .head(12).index.tolist()
)

df_sankey = df_inter[
    df_inter['inst_source'].isin(top12_inst) & df_inter['inst_target'].isin(top12_inst)
].copy()
df_sankey = df_sankey.sort_values('total_colabs', ascending=False).head(30)

# Crear nombres cortos
short = {n: (n[:30] + '…' if len(n) > 30 else n) for n in top12_inst}
node_labels = [short[n] for n in top12_inst]
node_map = {n: i for i, n in enumerate(top12_inst)}

sources = [node_map[r['inst_source']] for _, r in df_sankey.iterrows() if r['inst_source'] in node_map and r['inst_target'] in node_map]
targets = [node_map[r['inst_target']] for _, r in df_sankey.iterrows() if r['inst_source'] in node_map and r['inst_target'] in node_map]
values = [r['total_colabs'] for _, r in df_sankey.iterrows() if r['inst_source'] in node_map and r['inst_target'] in node_map]

node_colors = [PALETTE[i % len(PALETTE)] for i in range(len(node_labels))]
link_colors = [node_colors[s].replace(')', ',0.3)').replace('rgb', 'rgba') if 'rgb' in node_colors[s] else node_colors[s] + '40' for s in sources]

fig5 = go.Figure(go.Sankey(
    node=dict(
        pad=20, thickness=20,
        label=node_labels,
        color=node_colors,
        line=dict(width=0),
    ),
    link=dict(
        source=sources, target=targets, value=values,
        color=link_colors,
    ),
))

fig5.update_layout(
    template=dark_template, height=500,
    title='Flujos de colaboración inter-institucional',
)
fig5.show()

---
## ¿Qué encontramos?

La red científica mexicana es **densa en el centro, dispersa en los bordes**. Un núcleo de investigadores de la UNAM, CINVESTAV, Tec de Monterrey e IPN concentra la mayoría de conexiones. Los puentes entre instituciones son **pocos pero críticos** — un puñado de investigadores sostiene la conexión entre comunidades que de otra forma estarían aisladas.

La pregunta que queda abierta: **¿cómo se diseñan políticas que incentiven las conexiones correctas?**

---
*Datos: OpenAlex API · CC0 · Análisis: Python / NetworkX / Plotly*